# 1 · Preprocesamiento de datos

> **Tipo de ML:** `supervisado`

## Decisiones de diseño (procesado)

- **Una fila = la hora de MAYOR riesgo del día** (`select_risk_hour_row`), no la media de las 24 h: no diluir el pico térmico.
- **Estadísticas de la distribución diaria** (`heat_index_mean/std/min`, `horas_sobre_umbral`, y análogas de frío): distinguen un día de calor/frío **sostenido** de otro con el mismo pico pero **puntual**.
- **Features de persistencia entre días** (`_agregar_rezagos_temporales`: lags y medias móviles del pasado): el riesgo depende de la **racha**, no solo del día. Se calculan **estrictamente sobre el pasado** (shift por provincia) → sin fuga de datos aunque el split sea temporal.
- **Etiqueta por percentil de mortalidad POR PROVINCIA** (no global): una provincia pequeña no debe quedar siempre "seguro" por tener números absolutos de mortalidad bajos.
- **Split por FECHA** (no aleatorio): evita que días de la misma ola (correlacionados entre sí) queden repartidos entre train y test e inflen el rendimiento aparente.
- **Anti-fuga de datos**: se excluyen de `X` la mortalidad bruta de la que sale la etiqueta, la etiqueta de la otra clase, y los identificadores (`provincia`, `fecha`, `datetime`) — el modelo debe aprender de **física** (temperatura/humedad/viento), no de "dónde" ni "cuándo".

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import joblib

from climasafeai.data.make_dataset import load_data
from climasafeai.features.build_features import preprocess_data
from climasafeai.utils.paths import PROCESSED_DATA_DIR, ARTIFACTS_DIR
from climasafeai.utils.paths import RAW_DATA_DIR, FIGURES_DIR
from climasafeai.data.make_dataset import load_data, dataset_calor, cargar_era5_filtrado, cargar_provincias_unificadas, calcular_puntos_provincia  
from climasafeai.features.labels import asignar_clase_riesgo_calor, asignar_clase_riesgo_frio

## 2. Cargar datos crudos

In [2]:
provincias = cargar_provincias_unificadas()
puntos_por_provincia = calcular_puntos_provincia(provincias, col_nombre="NAMEUNIT")
df_eras = cargar_era5_filtrado(puntos_por_provincia)
df_eras.head(3)

--> Cargando y filtrando 126 archivos ERA5...


<xarray.Dataset> Size: 908B
Dimensions:     (valid_time: 3, punto: 3)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 24B 2016-01-01 ... 2016-01-01T02:...
    expver      (valid_time) <U4 48B '0001' '0001' '0001'
    latitude    (punto) float64 24B 43.25 43.75 42.5
    longitude   (punto) float64 24B -8.5 -7.75 -9.0
    provincia   (punto) <U44 528B 'A Coruña' 'A Coruña' 'A Coruña'
    tipo_punto  (punto) <U6 72B 'centro' 'norte' 'sur'
    number      int64 8B 0
Dimensions without coordinates: punto
Data variables:
    t2m         (valid_time, punto) float32 36B 284.2 285.4 ... 285.5 285.7
    d2m         (valid_time, punto) float32 36B 280.5 280.6 ... 280.1 282.0
    sp          (valid_time, punto) float32 36B 9.955e+04 ... 1.013e+05
    u10         (valid_time, punto) float32 36B -0.7338 0.4088 ... -0.9486
    v10         (valid_time, punto) float32 36B 6.08 5.216 7.805 ... 5.451 9.202
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-07-02T11:29 GRIB to CDM+CF via cfgrib-0.9.1...

In [3]:

DATA_FILE = RAW_DATA_DIR / "momo_data.csv"
try:
    df_momo = load_data(DATA_FILE)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Archivo no encontrado: {DATA_FILE}\n"
        "Coloca el CSV en data/raw/ y ajusta DATA_FILE arriba."
    )
df_momo.head(10)


--> Cargando datos desde /home/cacelas/Documentos/anfaia/ClimaSafeAI/data/raw/momo_data.csv...


    Datos cargados. Dimensiones: (7257600, 15)


,ambito,cod_ambito,cod_ine_ambito,nombre_ambito,cod_sexo,nombre_sexo,cod_gedad,nombre_gedad,fecha_defuncion,defunciones_observadas,defunciones_estimadas_base,defunciones_estimadas_base_q01,defunciones_estimadas_base_q99,defunciones_atrib_exc_temp,defunciones_atrib_def_temp
0,ccaa,AN,1.0,Andalucía,1,hombres,+65,edad >= 65,2015-01-01,108.88,83.25,63.0,105.0,0.0,5.50
1,ccaa,AN,1.0,Andalucía,1,hombres,+85,edad >= 85,2015-01-01,35.57,27.39,16.0,40.0,0.0,1.61
2,ccaa,AN,1.0,Andalucía,1,hombres,0-14,edad 0-14,2015-01-01,0.00,0.61,0.0,3.0,0.0,0.00
3,ccaa,AN,1.0,Andalucía,1,hombres,15-44,edad 15-44,2015-01-01,3.05,3.06,0.0,8.0,0.0,0.02
4,ccaa,AN,1.0,Andalucía,1,hombres,45-64,edad 45-64,2015-01-01,29.44,17.47,9.0,28.0,0.0,0.60
5,ccaa,AN,1.0,Andalucía,1,hombres,65-74,edad 65-74,2015-01-01,32.50,19.73,10.0,31.0,0.0,1.23
6,ccaa,AN,1.0,Andalucía,1,hombres,75-84,edad 75-84,2015-01-01,41.69,34.15,21.0,48.0,0.0,2.08
7,ccaa,AN,1.0,Andalucía,1,hombres,all,todos,2015-01-01,142.24,105.76,83.0,130.0,0.0,6.34
8,ccaa,AN,1.0,Andalucía,6,mujeres,+65,edad >= 65,2015-01-01,121.06,91.61,70.0,115.0,0.0,4.99
9,ccaa,AN,1.0,Andalucía,6,mujeres,+85,edad >= 85,2015-01-01,69.10,49.65,34.0,67.0,0.0,2.60


In [4]:
df = dataset_calor(df_momo, df_eras)
df = asignar_clase_riesgo_calor(df)

    ERA5: 24364630 filas (punto/hora) -> 4872926 (provincia/hora)
    ERA5: 4872926 filas (provincia/hora) -> 203043 (provincia/día)
    dataset_calor: 172395 filas (provincia/día)
    Suelo PELIGRO (>= 2.0 muertes): 884 días degradados a PRECAUCION
    Distribución de clases:
clase_riesgo_calor_label
seguro        154333
precaucion     10355
peligro         7707


## 3. Preprocesar


Indica la columna objetivo (`TARGET_COL`) y el tipo de scaler.


In [5]:
TARGET_COL = 'clase_riesgo_calor'   # la etiqueta 0/1/2, no la mortalidad bruta
SCALER_TYPE = 'standard'            # 'standard' | 'minmax'

X_train, X_test, y_train, y_test = preprocess_data(
    df,
    target_col=TARGET_COL,
    scaler_type=SCALER_TYPE,
    clase='calor',                  # explícito, aunque ya es el valor por defecto
)

print('X_train:', X_train.shape)
print('X_test: ', X_test.shape)
print('Balance clases (train):', pd.Series(y_train).value_counts(normalize=True).to_dict())

--> Preprocesando datos (clase='calor', target='clase_riesgo_calor', scaler='standard')...
    Columnas eliminadas: ['provincia', 'fecha', 'datetime', 'clase_riesgo_calor_label', 'defunciones_atrib_exc_temp']
    Split por fecha: train hasta 2024-05-22T00:00:00.000000000 | test desde 2024-05-23T00:00:00.000000000 (766 días distintos)
    feature_names_calor.joblib guardado (27 features)
    Scaler guardado → scaler_calor.joblib
    Train: (137925, 27) | Test: (34470, 27)
    Proporción clases (train): {0: 0.8982200471270618, 1: 0.0602573862606489, 2: 0.041522566612289286}
X_train: (137925, 27)
X_test:  (34470, 27)
Balance clases (train): {0: 0.8982200471270618, 1: 0.0602573862606489, 2: 0.041522566612289286}


## 4. Guardar dataset etiquetado (para validación temporal)

`preprocess_data()` guarda los conjuntos escalados que usa el entrenamiento normal. Aquí guardamos **además** el `df` etiquetado sin escalar (con `fecha`), que es lo que necesita la validación cruzada temporal del notebook de entrenamiento.

In [6]:
# ── Persistir el dataset ETIQUETADO (con 'fecha' + target) ──────────────────
# preprocess_data() de arriba ya guardó X/y escalados y SIN 'fecha' en
# data/processed/ (eso es lo que consume train_models()). Pero la validación
# cruzada temporal (temporal_cv.py) hace su PROPIO preprocesado por fold y
# necesita el df crudo etiquetado, con la columna 'fecha'. Lo guardamos aquí
# para poder cargarlo desde el notebook de entrenamiento (Parte B · validación).
df.to_parquet(PROCESSED_DATA_DIR / 'dataset_calor_labeled.parquet')
print(f"Dataset etiquetado guardado → dataset_calor_labeled.parquet ({df.shape[0]} filas)")

Dataset etiquetado guardado → dataset_calor_labeled.parquet (172395 filas)
